In [29]:
import numpy as np
import pandas as pd
from pathlib import Path

BASE       = Path.cwd().parent.parent / "ansys_data"
OUTPUT_DIR = Path.cwd().parent.parent / "data" / "raw"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [30]:
D_INNER  = 0.188
A_PIPE   = np.pi / 4 * D_INNER**2
RHO      = 1679.0
V_INLET  = 3.0
Q_INLET  = V_INLET * A_PIPE

X_A, X_B, X_C   = 5.0, 25.0, 45.0
X_BLOCKAGE       = 25.0     # all blockages at x=25m (Node B location)

# Normal CFD anchors (validated, t > 10s)
P_A_NORM = 76520.14
P_B_NORM = 38311.03
P_C_NORM =   189.49
V_NORM   =   2.9499
GRAD     = (P_A_NORM - P_C_NORM) / (X_C - X_A)   # 1908.3 Pa/m

# Normal transient shape (fitted to real CFD)
P_PEAK_A = 117246.90
P_PEAK_B =  58469.72
P_PEAK_C =    359.57
TAU_BASE =    1.80

N_STEPS  = 701
DT       = 0.1
TIMES    = np.round(np.arange(N_STEPS) * DT, 1)


In [31]:
BLOCKAGE_SCENARIOS = {
    "blockage_25": 0.75,   # 25% blocked → 75% area open  → label 4
    "blockage_50": 0.50,   # 50% blocked → 50% area open  → label 5
    "blockage_75": 0.25,   # 75% blocked → 25% area open  → label 6
}

LABEL = {"blockage_25": 2, "blockage_50": 2, "blockage_75": 2}
# all blockages = label 2 — severity in scenario column

# 3 blockage locations (Bernoulli + minor loss physics applies at any x)
# x=10m → Node A sees velocity spike; x=25m → Node B; x=40m → Node C
BLOCKAGE_POSITIONS = [10.0, 25.0, 40.0]

N_RUNS = 8


In [32]:

def blockage_steady_state(severity: str, x_block: float) -> dict:
    """
    Compute steady-state node pressures and velocities for a blockage
    at any axial position x_block.

    Key physics:
    - Whichever node is nearest the blockage sees the throat velocity spike
    - Nodes upstream of blockage: pressure rises (dP_blockage propagates back)
    - Nodes downstream: pressure reduced by permanent head loss
    - Fixed velocity inlet → upstream node velocities stay at V_INLET
    """
    area_frac = BLOCKAGE_SCENARIOS[severity]
    V_throat  = V_INLET / area_frac

    # Minor loss coefficients (Idelchik 2008)
    K_cont      = 0.5 * (1.0 - area_frac)
    K_exp       = (1.0 - area_frac) ** 2
    dP_blockage = 0.5 * RHO * (K_cont + K_exp) * V_throat**2

    # Normal pressure at blockage location (linear interpolation)
    P_at_block = P_A_NORM - GRAD * (x_block - X_A)

    def node_p(x_node):
        if x_node < x_block:
            # Upstream: pressure rises — blockage loss propagates back
            # Node closer to blockage feels more of it
            proximity = 1.0 - (x_block - x_node) / (x_block - X_A + 1e-9)
            base = P_A_NORM - GRAD * (x_node - X_A)
            return base + dP_blockage * 0.90 * max(proximity, 0.3)
        elif abs(x_node - x_block) < 5.0:
            # At blockage zone: reads between pre and post constriction
            base = P_A_NORM - GRAD * (x_node - X_A) + dP_blockage * 0.90
            return base - dP_blockage * 0.50
        else:
            # Downstream: normal gradient but shifted down by permanent loss
            base = P_A_NORM - GRAD * (x_node - X_A)
            return base - dP_blockage * 0.10

    def node_v(x_node):
        # Node nearest to blockage gets the throat velocity
        # "nearest" = within half the distance to next node
        node_positions = [X_A, X_B, X_C]
        nearest = min(node_positions, key=lambda x: abs(x - x_block))
        if x_node == nearest:
            return V_throat
        return V_INLET

    return dict(
        pA=max(node_p(X_A), 0.0),
        pB=max(node_p(X_B), 0.0),
        pC=max(node_p(X_C), 0.0),
        vA=node_v(X_A),
        vB=node_v(X_B),
        vC=node_v(X_C),
        dP=dP_blockage
    )


def build_transient(p_ss, p_peak, tau, rng):
    arr = np.empty(N_STEPS)
    noise = max(abs(p_ss) * 1e-4, 0.5)
    for i, t in enumerate(TIMES):
        if t == 0.0:
            arr[i] = 0.0
        else:
            arr[i] = p_ss + rng.normal(0, noise) + (p_peak - p_ss) * np.exp(-t / tau)
    return arr

In [33]:
def generate_blockage_run(severity: str, x_block: float,
                          run_id: int, rng: np.random.Generator) -> pd.DataFrame:
    ss = blockage_steady_state(severity, x_block)

    peak_scale = rng.uniform(0.98, 1.02)
    tau        = TAU_BASE * rng.uniform(0.95, 1.05)
    gs         = rng.normal(1.0, 0.0005)   # pump-speed jitter ±0.05%

    pA = build_transient(ss["pA"] * gs,
                         P_PEAK_A * peak_scale * (ss["pA"] / P_A_NORM),
                         tau, rng)
    pB = build_transient(ss["pB"] * gs,
                         P_PEAK_B * peak_scale * (ss["pB"] / P_B_NORM),
                         tau, rng)
    pC = build_transient(ss["pC"] * gs,
                         P_PEAK_C * peak_scale * max(ss["pC"] / P_C_NORM, 0.01),
                         tau * 0.8, rng)

    def vel_arr(v_ss):
        arr  = np.full(N_STEPS, v_ss * gs)
        arr += rng.normal(0, max(v_ss * 5e-5, 1e-4), N_STEPS)
        arr[0] = V_NORM * gs
        return arr

    return pd.DataFrame({
        "run_id":     run_id,
        "timestep":   np.arange(N_STEPS),
        "time_s":     TIMES,
        "pressure_A": pA,
        "pressure_B": pB,
        "pressure_C": pC,
        "velocity_A": vel_arr(ss["vA"]),
        "velocity_B": vel_arr(ss["vB"]),
        "velocity_C": vel_arr(ss["vC"]),
        "label":      LABEL[severity],
        "scenario":   severity,
    })

In [34]:
rng = np.random.default_rng(seed=42)
all_runs  = []
global_id = 0

for severity in ["blockage_25", "blockage_50", "blockage_75"]:
    for x_block in BLOCKAGE_POSITIONS:
        for _ in range(N_RUNS):
            all_runs.append(generate_blockage_run(severity, x_block, global_id, rng))
            global_id += 1

blockage_df = pd.concat(all_runs, ignore_index=True)
out_path = OUTPUT_DIR / "blockage_combined.csv"
blockage_df.to_csv(out_path, index=False, float_format="%.6f")

print("=" * 65)
print(f"  Saved  →  {out_path}")
print(f"  Total rows : {len(blockage_df):,}")
print(f"  Total runs : {global_id}  ({N_RUNS} per severity × location)")
print("=" * 65)
print(f"\n{'Scenario':<14} {'label':>5}  {'SS pA':>9}  {'SS pB':>9}  {'SS pC':>8}  {'SS vB':>7}  {'SS vC':>7}")
print("-" * 65)
ss_df = blockage_df[blockage_df["time_s"] >= 10]
for sc in ["blockage_25", "blockage_50", "blockage_75"]:
    r0  = ss_df[(ss_df["scenario"] == sc) &
                (ss_df["run_id"]   == ss_df[ss_df["scenario"]==sc]["run_id"].min())]
    lbl = LABEL[sc]
    print(f"{sc:<14}  {lbl:>5}  "
          f"{r0['pressure_A'].mean():>9.1f}  "
          f"{r0['pressure_B'].mean():>9.1f}  "
          f"{r0['pressure_C'].mean():>8.2f}  "
          f"{r0['velocity_B'].mean():>7.4f}  "
          f"{r0['velocity_C'].mean():>7.4f}")
print("=" * 65)
print(f"\nNormal ref:    label=0   pA=76520   pB=38311   pC=189.49  vB=2.9532  vC=2.9417")


  Saved  →  /home/local-host/IdeaProjects/ai-pipeline-leak-detection/ml_service/data/raw/blockage_combined.csv
  Total rows : 50,472
  Total runs : 72  (8 per severity × location)

Scenario       label      SS pA      SS pB     SS pC    SS vB    SS vC
-----------------------------------------------------------------
blockage_25         2    77233.7    38119.6     -0.07   3.0011   3.0011
blockage_50         2    80584.7    36836.6     -0.02   2.9992   2.9992
blockage_75         2   107127.2    27023.5     -0.03   2.9999   3.0000

Normal ref:    label=0   pA=76520   pB=38311   pC=189.49  vB=2.9532  vC=2.9417


In [35]:
blockage_df.shape

(50472, 11)